# Phần 6.2 — Pilot một epoch và ngân sách thí nghiệm
**2A202602810 · Nguyễn Thái Anh**

Chọn kernel **DeepWeeds (.venv)** trên máy có GPU, chạy lần lượt hoặc Run All.
Notebook chạy tuần tự 5 backbone ImageNet pretrained, mỗi model **một epoch train đầy đủ theo DataLoader và toàn bộ val**.
Giữ lịch T00 12 epoch, warmup 1 epoch; gọi run với stop_after_epoch=1, thay vì đổi epochs thành 1.
Không dùng test để đánh giá. Chỉ đọc CSV test cho audit split nguyên bản.

Output: checkpoint/config/history trong runs/pilot62; prediction val, bảng và JSON trong code/validation/pilot62;
hình trong curves/pilot62. Pilot có ID riêng, **không phải B01–B05 hay kết quả 12 epoch**.
Checkpoint last lưu đủ trạng thái để resume cùng cấu hình. Trọng số public được cache trong repo, không cần đăng nhập Hugging Face.

## 1. Môi trường và đường dẫn
Notebook tìm gốc repo từ thư mục hiện tại; đặt ROOT_HINT nếu mở từ nơi khác. Sau khi sửa môi trường/kernel, restart kernel trước Run All.

In [ ]:
import os, sys, json, gc, time, traceback, math
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import asdict, replace

ROOT_HINT = None
ROOT = Path(ROOT_HINT).expanduser().resolve() if ROOT_HINT else next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / "eval.py").is_file() and (p / "starter").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Đặt ROOT_HINT tới gốc repo chứa eval.py và starter/")
SUBMISSION = ROOT / "submissions/2A202602810_NguyenThaiAnh"
CODE = SUBMISSION / "code"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HOME"] = str(ROOT / ".cache/huggingface")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(CODE))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
import dataset
import train as trainer
from eval import read_pred

assert Path(trainer.__file__).resolve() == (CODE / "train.py").resolve()
torch.set_num_threads(4)
pd.set_option("display.max_columns", 30)
print("ROOT:", ROOT)
print("Python:", sys.executable)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Cấu hình pilot và phiên chạy
Mặc định Run All sẽ chạy pilot trên GPU. RUN_PILOTS=False chỉ đọc các kết quả đã có.
Giữ SESSION_ID khi mở lại notebook để tái dùng pilot đã hoàn thành; đổi SESSION_ID để đo phiên mới.
Không tự ghi đè run đã có. Run gián đoạn có last.pt sẽ resume tới epoch 1; nếu chưa có checkpoint, dùng SESSION_ID mới.

PLANNED_EPOCHS là chân trời scheduler của thí nghiệm dự kiến (12), không phải số epoch sẽ chạy trong notebook (1).
Batch chung 64; nếu một model OOM, giảm batch chung và đo lại toàn bộ bằng SESSION_ID mới.

In [ ]:
SESSION_ID = "P62_session01"
RUN_PILOTS = True
PLANNED_EPOCHS = 12
BATCH_SIZE = 64
NUM_WORKERS = 2
SEED = 0
BACKBONES = [
    ("01", "resnet50"),
    ("02", "resnext50_32x4d"),
    ("03", "convnext_tiny"),
    ("04", "deit_small_patch16_224"),
    ("05", "efficientnet_b0"),
]
assert SESSION_ID and SESSION_ID not in {".", ".."} and Path(SESSION_ID).name == SESSION_ID
assert PLANNED_EPOCHS >= 2 and BATCH_SIZE > 0 and NUM_WORKERS >= 0

TABLE_DIR = CODE / "validation/pilot62" / SESSION_ID
PRED_DIR = TABLE_DIR / "predictions"
FIG_DIR = SUBMISSION / "curves/pilot62" / SESSION_ID
for path in (TABLE_DIR, PRED_DIR, FIG_DIR):
    path.mkdir(parents=True, exist_ok=True)

base_cfg = trainer.Config(
    exp_id=SESSION_ID, seed=SEED, epochs=PLANNED_EPOCHS,
    init="finetune", img_size=224, aug="basic", loss="ce",
    mix=None, label_smoothing=0, sampler=None, ema_decay=None,
    lr_backbone=1e-4, lr_head=1e-3, weight_decay=.05, warmup_epochs=1,
    batch_size=BATCH_SIZE, amp=True, num_workers=NUM_WORKERS,
    device="cuda:0", deterministic=True, save_test_predictions=False,
    images_dir=str(ROOT / "data"), labels_dir=str(ROOT / "data/labels"),
    out_dir=str(ROOT / "runs/pilot62"), pred_dir=str(PRED_DIR),
    curves_dir=str(FIG_DIR), cache_dir=str(ROOT / ".cache/models"))
trainer.validate_config(base_cfg)
pilot_configs = [replace(base_cfg, exp_id=f"{SESSION_ID}_{index}", backbone=name)
                 for index, name in BACKBONES]
display(pd.DataFrame([{"exp_id": c.exp_id, "backbone": c.backbone,
                      "scheduled_epochs": c.epochs, "pilot_epochs": 1,
                      "batch_size": c.batch_size, "seed": c.seed}
                     for c in pilot_configs]))

## 3. Điều kiện đầu vào và audit
Sanity checks đã đạt phải có summary.json. Giữ nguyên fold 0, hashes và các nhãn split.
Train mặc định drop_last=True: batch 64 xử lý 10.496/10.501 ảnh trong một epoch, bỏ batch cuối 5 ảnh;
đây là hành vi T00 hiện có, không lọc/sửa CSV. Val luôn dùng đủ 3.501 ảnh và giữ batch cuối.

In [ ]:
sanity_path = CODE / "validation/sanity/summary.json"
if not sanity_path.exists():
    raise FileNotFoundError("Chạy code/sanity_checks.py trước notebook pilot")
sanity = json.loads(sanity_path.read_text())
assert sanity["all_passed"], "Sanity chưa đạt"
train_df, val_df, test_df = dataset.load_split(base_cfg.labels_dir, fold=0)
reference = pd.read_csv(Path(base_cfg.labels_dir) / "labels.csv")
split_report = dataset.check_split(train_df, val_df, test_df, base_cfg.images_dir, reference_df=reference)
import hashlib
hashes = {name: hashlib.sha256((Path(base_cfg.labels_dir) / name).read_bytes()).hexdigest()
          for name in ("labels.csv", "train_subset0.csv", "val_subset0.csv", "test_subset0.csv")}
trainer._verify_provenance(hashes, 0)
trainer.write_json(TABLE_DIR / "split_check.json", split_report)
trainer.write_json(TABLE_DIR / "split_hashes.json", hashes)
if RUN_PILOTS and not torch.cuda.is_available():
    raise RuntimeError("Mở bằng kernel .venv trên terminal host có CUDA; sandbox không có GPU.")
display(split_report)

## 4. Hàm chạy và kiểm tra một pilot
Đo epoch train+val bao gồm DataLoader/preprocessing/forward/backward và tính metric, không bao gồm download,
khởi tạo model, GMAC profiling, checkpoint, vẽ hình hay lần export val cuối.
wall_seconds đo toàn bộ lời gọi run, gồm các phần trên; không nhân wall_seconds với 12.
Peak VRAM được reset sau profiling và đo trên epoch train+val; ghi cả allocated và reserved.
Throughput tính từ số ảnh thực xử lý. Clock GPU, ổ đĩa và tải desktop có thể ảnh hưởng số đo.

In [ ]:
def check_pilot_outputs(cfg):
    path = trainer.run_dir(cfg)
    required = ["config.json", "environment.json", "split_check.json", "split_hashes.json",
                "model.json", "mac_profile.json", "history.csv", "best.pt", "last.pt",
                "val_logits.npy", "val_labels.npy", "val_filenames.npy", "summary.json"]
    missing = [name for name in required if not (path / name).is_file()]
    if missing:
        raise FileNotFoundError(f"{path}: thiếu {missing}")
    summary = json.loads((path / "summary.json").read_text())
    history = pd.read_csv(path / "history.csv")
    saved_cfg = json.loads((path / "config.json").read_text())
    requested_cfg = asdict(cfg)
    saved_cfg.pop("resume", None)
    requested_cfg.pop("resume", None)
    assert saved_cfg == requested_cfg, "Config khác phiên đã lưu; chọn SESSION_ID mới."
    assert summary["completed_epochs"] == 1 and summary["scheduled_epochs"] == PLANNED_EPOCHS
    assert not summary["training_complete"] and summary["stop_after_epoch"] == 1
    assert len(history) == 1 and int(history.iloc[0]["epoch"]) == 1
    assert "test" not in summary and not (path / "test_started.json").exists()
    assert not trainer.pred_path(cfg, "test").exists()
    predictions = read_pred(str(trainer.pred_path(cfg, "val")))
    assert predictions.filenames.tolist() == val_df["Filename"].tolist()
    np.testing.assert_array_equal(predictions.y_true, val_df["Label"].to_numpy())
    assert predictions.probs.shape == (len(val_df), 9)
    np.testing.assert_allclose(predictions.probs.sum(1), 1, atol=1e-6)
    model_metadata = json.loads((path / "model.json").read_text())
    assert model_metadata["pretrained_loaded"]
    curve_path = Path(cfg.curves_dir) / f"{cfg.exp_id}_seed{cfg.seed}_{cfg.backbone}.png"
    assert curve_path.is_file()
    return summary, history.iloc[0].to_dict(), model_metadata

def pilot_one(cfg):
    path = trainer.run_dir(cfg)
    measured_path = TABLE_DIR / f"{cfg.exp_id}_measurement.json"
    reuse = (path / "summary.json").exists() and measured_path.exists()
    if reuse:
        summary, epoch, metadata = check_pilot_outputs(cfg)
        previous = json.loads(measured_path.read_text())
        assert previous["status"] == "completed"
        print("Đọc pilot đã hoàn thành:", cfg.backbone)
        return previous
    if not RUN_PILOTS:
        return {"exp_id": cfg.exp_id, "backbone": cfg.backbone, "status": "not_run"}
    if (path / "config.json").exists():
        if not (path / "last.pt").is_file():
            raise FileExistsError(f"{path}: chưa có last.pt; đổi SESSION_ID để chạy lại, không ghi đè.")
        cfg = replace(cfg, resume=True)
    print(f"Pilot {cfg.backbone}: một epoch train + toàn bộ val", flush=True)
    torch.cuda.synchronize(0)
    start = time.perf_counter()
    trainer.run(cfg, stop_after_epoch=1)
    torch.cuda.synchronize(0)
    wall_seconds = time.perf_counter() - start
    summary, epoch, metadata = check_pilot_outputs(cfg)
    val_count = summary["val"]["n"]
    row = {
        "exp_id": cfg.exp_id, "backbone": cfg.backbone, "status": "completed",
        "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
        "gpu": torch.cuda.get_device_name(0), "seed": cfg.seed, "batch_size": cfg.batch_size,
        "img_size": cfg.img_size, "scheduled_epochs": cfg.epochs, "completed_epochs": 1,
        "train_amp": True, "val_dtype": "FP32", "num_workers": cfg.num_workers,
        "train_images": int(epoch["train_images"]), "train_rows_total": len(train_df),
        "val_images": val_count, "optimizer_updates": int(epoch["optimizer_updates"]),
        "skipped_updates": int(epoch["skipped_updates"]),
        "train_seconds": epoch["train_seconds"], "val_seconds": epoch["val_seconds"],
        "epoch_train_val_seconds": epoch["epoch_train_val_seconds"],
        "wall_seconds": wall_seconds, "wall_is_resumed_invocation": cfg.resume,
        "peak_allocated_gib": epoch["peak_allocated_gib"], "peak_reserved_gib": epoch["peak_reserved_gib"],
        "train_images_per_second": epoch["train_images"] / epoch["train_seconds"],
        "val_images_per_second": val_count / epoch["val_seconds"],
        "estimated_minutes_full_run": cfg.epochs * epoch["epoch_train_val_seconds"] / 60,
        "params_m": summary["params_m"], "gmacs": summary["gmacs"],
        "pretrained_tag": metadata["pretrained_cfg"].get("tag"),
        "val_macro_f1_after_ONE_epoch": summary["val"]["macro_f1"],
        "outputs_verified": True, "run_path": str(path.relative_to(ROOT)),
        "scope": "1 measured full epoch; extrapolation excludes per-epoch checkpoint/plot overhead",
    }
    trainer.write_json(measured_path, row)
    return row

## 5. Chạy tuần tự cả 5 backbone
Giữ model lỗi ở trạng thái failed/OOM, không gán metric giả. Kết quả được lưu sau từng model.
Nếu OOM, dùng batch chung nhỏ hơn và SESSION_ID mới để đo lại cả 5.
Không dùng macro-F1 của pilot một epoch để chọn backbone; so sánh B01–B05 sẽ chạy đủ protocol.

In [ ]:
pilot_rows = []
for cfg in pilot_configs:
    try:
        row = pilot_one(cfg)
    except Exception as exc:
        status = "OOM" if isinstance(exc, torch.cuda.OutOfMemoryError) else "failed"
        row = {"exp_id": cfg.exp_id, "backbone": cfg.backbone,
               "status": status, "error": f"{type(exc).__name__}: {exc}"}
        traceback.print_exc()
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    pilot_rows.append(row)
    trainer.write_json(TABLE_DIR / "pilot_results.json", pilot_rows)
    pd.DataFrame(pilot_rows).to_csv(TABLE_DIR / "pilot_results.csv", index=False)

pilot_table = pd.DataFrame(pilot_rows)
display(pilot_table)
print("Kết quả:", TABLE_DIR / "pilot_results.csv")

## 6. Tổng hợp thời gian, VRAM và throughput
Chỉ dùng các pilot đã hoàn thành và kiểm tra output. allocated là tensor đang dùng; reserved là bộ nhớ allocator đã giữ.

In [ ]:
completed = pilot_table.loc[pilot_table["status"] == "completed"].copy()
if completed.empty:
    print("Chưa có pilot hoàn thành. Bật RUN_PILOTS trên GPU rồi chạy lại.")
else:
    columns = ["backbone", "batch_size", "train_seconds", "val_seconds", "epoch_train_val_seconds",
               "wall_seconds", "peak_allocated_gib", "peak_reserved_gib",
               "train_images_per_second", "val_images_per_second",
               "estimated_minutes_full_run", "skipped_updates"]
    display(completed[columns].round(3))
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    labels = completed["backbone"].tolist()
    positions = np.arange(len(labels))
    axes[0].bar(positions, completed["train_seconds"], label="Train")
    axes[0].bar(positions, completed["val_seconds"], bottom=completed["train_seconds"], label="Val")
    axes[0].set(ylabel="Seconds, measured first full epoch", title="Train / val runtime")
    axes[1].bar(positions-.18, completed["peak_allocated_gib"], .36, label="Allocated")
    axes[1].bar(positions+.18, completed["peak_reserved_gib"], .36, label="Reserved")
    axes[1].set(ylabel="GiB", title="Peak VRAM during train + val")
    for ax in axes:
        ax.set_xticks(positions, labels, rotation=25, ha="right")
        ax.legend()
        ax.grid(axis="y", alpha=.2)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "pilot_runtime_vram.png", dpi=160)
    plt.show()

## 7. Ngân sách toàn bài — tham số kế hoạch
Tách 5 backbone, ablation, final ≥3 seed, baseline ≥3 seed, suy luận/export và dự phòng.
PLANNING_BACKBONE chỉ phục vụ tính thời gian, **chưa phải backbone thắng trên val**.
T00–T08 mặc định 9 run; có thể giảm số run mới nếu một baseline trước đó cùng protocol được tái dùng có bằng chứng.
Các recipe có mix/EMA/augmentation khác có thể tốn thời gian khác T00; nhân ABLATION_TIME_FACTOR / FINAL_TIME_FACTOR.
INFERENCE_EXPORT_MINUTES và DOWNLOAD_SETUP_MINUTES là thời gian **dành chỗ trong kế hoạch**, không phải số đo.
Epoch đầu chứa startup và warmup; ngoại suy 12 epoch vẫn là ước tính. MARGIN_RATIO dành cho checkpoint/plots, restart và biến động.

In [ ]:
PLANNING_BACKBONE = "convnext_tiny"
N_ABLATION_RUNS = 9
N_FINAL_SEEDS = 3
N_BASELINE_SEEDS = 3
ABLATION_TIME_FACTOR = 1.0
FINAL_TIME_FACTOR = 1.0
INFERENCE_EXPORT_MINUTES = 30.0
DOWNLOAD_SETUP_MINUTES = 10.0
MARGIN_RATIO = 0.30
assert N_FINAL_SEEDS >= 3 and N_BASELINE_SEEDS >= 3
assert N_ABLATION_RUNS >= 0 and min(ABLATION_TIME_FACTOR, FINAL_TIME_FACTOR) > 0
assert min(INFERENCE_EXPORT_MINUTES, DOWNLOAD_SETUP_MINUTES, MARGIN_RATIO) >= 0

budget = None
all_five_complete = len(completed) == len(BACKBONES) and completed["backbone"].nunique() == len(BACKBONES)
if not all_five_complete:
    print("Chưa tính ngân sách toàn bài: cần đo đủ cả 5 backbone.")
elif PLANNING_BACKBONE not in completed["backbone"].values:
    raise ValueError("PLANNING_BACKBONE chưa có số đo")
else:
    epoch_seconds = float(completed.set_index("backbone").loc[PLANNING_BACKBONE, "epoch_train_val_seconds"])
    baseline_run_minutes = epoch_seconds * PLANNED_EPOCHS / 60
    components = [
        {"phase": "5 backbone", "runs": len(BACKBONES),
         "minutes": float(completed["estimated_minutes_full_run"].sum())},
        {"phase": "Ablation T00–T08", "runs": N_ABLATION_RUNS,
         "minutes": N_ABLATION_RUNS * baseline_run_minutes * ABLATION_TIME_FACTOR},
        {"phase": "Final", "runs": N_FINAL_SEEDS,
         "minutes": N_FINAL_SEEDS * baseline_run_minutes * FINAL_TIME_FACTOR},
        {"phase": "Baseline chung kết", "runs": N_BASELINE_SEEDS,
         "minutes": N_BASELINE_SEEDS * baseline_run_minutes},
        {"phase": "Suy luận / benchmark / export (dành chỗ)", "runs": None,
         "minutes": INFERENCE_EXPORT_MINUTES},
        {"phase": "Download / setup (dành chỗ)", "runs": None,
         "minutes": DOWNLOAD_SETUP_MINUTES},
    ]
    subtotal = sum(item["minutes"] for item in components)
    components.append({"phase": f"Dự phòng {MARGIN_RATIO:.0%}", "runs": None, "minutes": subtotal * MARGIN_RATIO})
    total = subtotal * (1 + MARGIN_RATIO)
    budget_table = pd.DataFrame(components)
    display(budget_table.round(2))
    print(f"Tổng kế hoạch: {total:.1f} phút ≈ {total/60:.2f} giờ.")
    budget = {"scope": "planning estimate, not measured full experiments", "planning_backbone": PLANNING_BACKBONE,
              "planned_epochs": PLANNED_EPOCHS, "n_ablation_runs": N_ABLATION_RUNS,
              "n_final_seeds": N_FINAL_SEEDS, "n_baseline_seeds": N_BASELINE_SEEDS,
              "ablation_time_factor": ABLATION_TIME_FACTOR, "final_time_factor": FINAL_TIME_FACTOR,
              "margin_ratio": MARGIN_RATIO, "components": components, "total_minutes": total}
    trainer.write_json(TABLE_DIR / "budget.json", budget)
    budget_table.to_csv(TABLE_DIR / "budget.csv", index=False)

## 8. Ghi kết luận và bước tiếp theo
Mục 6.2 chỉ hoàn tất khi đủ 5 pilot, output hợp lệ và có ngân sách toàn bài.
Nếu có skipped optimizer updates, kiểm tra AMP/gradient trước khi chạy dài.
Trước B01–B05, chốt epoch/batch chung; không gộp val vào train và không bật test.

In [ ]:
pilot_complete = all_five_complete and budget is not None
skipped_updates = int(completed["skipped_updates"].sum()) if not completed.empty else None
ready_for_backbones = pilot_complete and skipped_updates == 0
notes = [
    "# Phần 6.2 — trạng thái pilot và ngân sách",
    "",
    f"- SESSION_ID: {SESSION_ID}; pilot hoàn thành {len(completed)}/{len(BACKBONES)}.",
    f"- Phần 6.2 hoàn tất: {pilot_complete}; sẵn sàng chuyển sang backbone: {ready_for_backbones}.",
    f"- Horizon scheduler: {PLANNED_EPOCHS} epoch; thực chạy: 1 epoch/model.",
    f"- Batch chung: {BATCH_SIZE}; seed: {SEED}; skipped updates: {skipped_updates}.",
    "- Không đánh giá test; F1 pilot chỉ là kiểm tra, không chọn backbone.",
    "- Epoch timing loại trừ download/profiling/checkpoint/plot và export val lần cuối.",
    "- wall_seconds bao gồm toàn bộ lời gọi run; khi resume là thời gian của lời gọi tiếp tục.",
]
if budget is not None:
    notes.append(f"- Ngân sách dự kiến: {budget['total_minutes']:.1f} phút; xem budget.json cho các giả định.")
notes += ["", "## Tiếp theo",
          "Chạy B01–B05 theo cùng T00 đủ epoch, chọn backbone bằng macro-F1 val.",
          "Không ghi các pilot này vào bảng kết quả 12 epoch."]
(TABLE_DIR / "pilot_notes.md").write_text("\n".join(notes) + "\n", encoding="utf-8")
print("\n".join(notes))

## 9. Resume pilot thành thí nghiệm đủ epoch — tùy chọn, chạy riêng
Khi có last.pt, dùng chính config đã lưu và chỉ đổi resume=True.
Ví dụ trong một cell mới: cfg = pilot_configs[0]; trainer.run(replace(cfg, resume=True)).
Lệnh này **chạy các epoch 2–12**, không phải pilot; không đưa vào Run All của notebook hiện tại.
Không đổi epoch/batch/LR/scheduler giữa pilot và resume; mã run sẽ từ chối config khác.
Nếu resume đủ epoch, dùng ID đó như một run đủ protocol có provenance, hoặc tạo ID B01 mới và train lại từ đầu.
Để giữ bảng pilot một epoch, không chạy lại notebook trên ID đã tiếp tục đủ epoch; chọn SESSION_ID mới.